# PT-03 — Direct Preference Optimization (DPO)

**Serie** : Post-Training SOTA 2024-2025 (Epic #1742, sub-issue #1756)
**Pre-requis** : PT-01 (intro) et PT-02 (SFT baseline) recommandes
**Objectifs pedagogiques** :
1. Comprendre la formule du loss DPO et son lien avec le modele de Bradley-Terry
2. Savoir pourquoi DPO contourne le reward modeling explicite de RLHF classique
3. Implementer `trl.DPOTrainer` sur des paires chosen/rejected
4. Evaluer la preference accuracy : frequence ou le modele prefere chosen > rejected
5. Identifier les 4 pieges classiques du DPO en pratique

**Reference cle** : Rafailov et al., "Direct Preference Optimization: Your Language Model is Secretly a Reward Model" (NeurIPS 2023), arXiv:2305.18290

## 1. Le loss DPO — de RLHF a l'optimisation directe

### Rappel RLHF classique (3 etapes)

| Etape | Objectif | Cout |
|-------|----------|------|
| 1. SFT | Pre-entrainer le modele sur des instructions | GPU moderate |
| 2. Reward Model | Entrainer un modele a scorer chosen > rejected | GPU eleve (2 modeles) |
| 3. PPO | Optimiser la policy contre le reward model + KL penalty | GPU tres eleve (4 modeles) |

Le DPO **elimine l'etape 2** (reward model) et **remplace l'etape 3** par un loss classification binaire direct.

### Formule du loss DPO

$$\mathcal{L}_{\text{DPO}}(\theta) = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma \left( \beta \left( \log \frac{\pi_\theta(y_w | x)}{\pi_{\text{ref}}(y_w | x)} - \log \frac{\pi_\theta(y_l | x)}{\pi_{\text{ref}}(y_l | x)} \right) \right) \right]$$

**Decomposition** :

- $y_w$ = **chosen** (reponse preferee), $y_l$ = **rejected** (reponse rejetee)
- $\pi_\theta$ = policy entrainee, $\pi_{\text{ref}}$ = reference (modele SFT gele)
- $\beta$ = coefficient de regularisation KL (typique 0.1)
- $\sigma$ = sigmoid

**Intuition** : le loss maximise l'ecart de log-probabilites entre chosen et rejected, **normalise par la reference**. Le modele apprend a preferer chosen SANS jamais calculer explicitement un reward.

### Lien Bradley-Terry → implicit reward

Le modele de Bradley-Terry modelise la probabilite que $y_w$ soit prefere a $y_l$ :

$$P(y_w \succ y_l | x) = \sigma\left( r(x, y_w) - r(x, y_l) \right)$$

Le DPO montre que le **reward implicite** est :

$$r(x, y) = \beta \log \frac{\pi_\theta(y | x)}{\pi_{\text{ref}}(y | x)}$$

Le modele de langage **EST** le reward model — d'ou le titre du papier.

## 2. Pourquoi DPO contourne le reward modeling

| Critere | RLHF (PPO) | DPO |
|---------|------------|-----|
| Nombre de modeles en memoire | 4 (policy, ref, reward, critic) | 2 (policy + ref) |
| Instabilite d'entrainement | Elevee (reward hacking, mode collapse) | Faible (classification binaire) |
| Cout GPU | Tres eleve | Moderate |
| Besoin de donnees | Paires + reward model | Paires seulement |
| Complexite implementation | Elevee (clip, advantage, GAE) | Faible (cross-entropy modifiee) |

**Trade-off** : DPO est plus simple et stable, mais moins expressif que RLHF avec un reward model bien entraine. En pratique, DPO match ou surpasse RLHF sur la plupart des benchmarks d'alignement (Zephyr, Tulu 2).

**Cas ou RLHF reste preferable** : si le reward model capture des signaux complexes que les paires chosen/rejected ne couvrent pas bien.

## 3. Verification de l'environnement

DPO charge **2 modeles** en memoire (policy + reference), donc la contrainte GPU est plus forte que pour SFT. Avec quantization 4-bit et LoRA, on reste sous 6 Go VRAM.

In [1]:
import sys
import os
import platform
import warnings

# Les avertissements de deprecation de bitsandbytes/torch incluent le chemin
# d'installation machine (fuite de path). On les supprime cote source pour des sorties propres.
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"Python : {sys.version}")
print(f"Plateforme : {platform.platform()}")
print(f"Repertoire de travail : {os.getcwd()}")

# Flag de controle : execution complete activee (GPU requis pour les cellules d'entrainement).
# La garde CUDA en cellule [20] protege encore les machines CPU-only.
LOAD_MODEL_AND_TRAIN = True
print(f"\nLOAD_MODEL_AND_TRAIN = {LOAD_MODEL_AND_TRAIN}")
print("Mode : EXECUTION RELLE (entrainement DPO sur GPU)")

Python : 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
Plateforme : Windows-11-10.0.26200-SP0
Repertoire de travail : C:\dev\CoursIA\MyIA.AI.Notebooks\PostTraining

LOAD_MODEL_AND_TRAIN = False
(Mode CPU-safe : les cellules d'entrainement seront skippees)
(Mettre LOAD_MODEL_AND_TRAIN = True sur machine avec GPU pour execution reelle)


In [2]:
import torch

CUDA_AVAILABLE = torch.cuda.is_available()
if CUDA_AVAILABLE:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"CUDA disponible : {gpu_name} ({gpu_mem:.1f} Go)")
else:
    print("CUDA non disponible (CPU uniquement)")
    print("L'entrainement DPO sera skippe (mode CPU-safe)")

CUDA disponible : NVIDIA GeForce RTX 3070 Laptop GPU (8.0 Go)


## 4. Dataset : paires chosen/rejected

Le DPO necessite des **paires de preference** — pas des instructions seules. Chaque exemple contient :
- Un prompt `x`
- Une reponse preferee `y_w` (chosen)
- Une reponse rejetee `y_l` (rejected)

Source : `HuggingFaceH4/ultrafeedback_binarized` (train_prefs, 50 exemples pour le demo).

In [3]:
from datasets import load_dataset
import datasets as _datasets_lib

# En mode hors-ligne, datasets affiche le chemin absolu du cache (fuite de path).
# On reduit la verbosite au niveau ERREUR pour des sorties propres.
_datasets_lib.logging.set_verbosity_error()

# Charger les preferences binarisees (chosen vs rejected)
dataset_dpo = load_dataset(
    "HuggingFaceH4/ultrafeedback_binarized",
    split="train_prefs[:50]"
)

print(f"Dataset DPO charge : {len(dataset_dpo)} exemples")
print(f"Colonnes : {dataset_dpo.column_names}")

# TRL 1.7.0 attend un format STANDARD (prompt/chosen/rejected en chaines), mais
# ultrafeedback_binarized stocke chosen/rejected en format conversationnel (listes
# de messages). On extrait le contenu assistant en chaine.
def _extraire_contenu(dataset):
    return dataset.map(
        lambda ex: {
            "prompt": ex["prompt"],
            "chosen": ex["chosen"][-1]["content"] if isinstance(ex["chosen"], list) else ex["chosen"],
            "rejected": ex["rejected"][-1]["content"] if isinstance(ex["rejected"], list) else ex["rejected"],
        },
        remove_columns=dataset.column_names,
    )


dataset_dpo = _extraire_contenu(dataset_dpo)
print(f"Format standardise (str) : {len(dataset_dpo)} paires pretes pour DPOTrainer.")


Dataset DPO charge : 50 exemples
Colonnes : ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected']


### Exercice 1 : Analyser la qualite des paires chosen/rejected

Le dataset `ultrafeedback_binarized` contient des paires de preferences, mais leur qualite varie. L'objectif est d'implementer une fonction `analyser_paires` qui calcule des statistiques sur les paires : nombre de paires ou le score chosen est effectivement superieur au score rejected, et identification des paires ambigues (scores egaux ou proches).

**Objectif** : ecrire une fonction qui parcourt le dataset et retourne un rapport sur la coherence des annotations (chosen devrait avoir un score superieur a rejected).

**Indices** :
- # Etape 1 : Comparer `score_chosen` et `score_rejected` pour chaque exemple
- # Etape 2 : Compter les paires coherentes (chosen > rejected), egales et incoherentes (chosen < rejected)
- # Indice : un dataset de bonne qualite devrait avoir majoritairement des paires coherentes

In [1]:
def analyser_paires(dataset) -> dict:
    # TODO etudiant : analyser la coherence des paires chosen/rejected
    
    coherentes = 0    # chosen_score > rejected_score
    egales = 0        # scores identiques
    incoherentes = 0  # chosen_score < rejected_score
    
    for exemple in dataset:
        # Etape 1 : extraire les scores
        score_c = exemple.get("score_chosen", 0)
        score_r = exemple.get("score_rejected", 0)
        
        # Etape 2 : classifier la paire
        pass  # TODO etudiant : incrementer le bon compteur
    
    return {
        "coherentes": coherentes,
        "egales": egales,
        "incoherentes": incoherentes,
        "total": len(dataset),
    }

print("Exercice a completer : analyse paires")

Exercice a completer


In [4]:
# Apercu d'une paire chosen/rejected
sample = dataset_dpo[0]

print("=" * 60)
print("PROMPT (debut) :")
print(sample['prompt'][:200] + "...")
print()
print("CHOSEN (debut) :")
chosen_text = sample['chosen'][0]['content'] if isinstance(sample['chosen'], list) else str(sample['chosen'])[:200]
print(chosen_text[:200] + "...")
print()
print("REJECTED (debut) :")
rejected_text = sample['rejected'][0]['content'] if isinstance(sample['rejected'], list) else str(sample['rejected'])[:200]
print(rejected_text[:200] + "...")
print("=" * 60)

PROMPT (debut) :
how can i develop a habit of drawing daily...

CHOSEN (debut) :
how can i develop a habit of drawing daily...

REJECTED (debut) :
how can i develop a habit of drawing daily...


## 5. Format ChatML du template Qwen2.5

Le DPO necessite le meme format de chat que le SFT (PT-02). La tokenisation applique le template automatiquement.

Structure d'une conversation :
```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
{prompt}<|im_end|>
<|im_start|>assistant
{response}<|im_end|>
```

`trl.DPOTrainer` gere la tokenisation automatiquement via le chat template du tokenizer.

In [5]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen3.5-0.8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Verifier que le chat template est disponible
if tokenizer.chat_template:
    print(f"Chat template trouve (longueur : {len(tokenizer.chat_template)} chars)")
    # Apercu du template
    print(f"Debut du template : {tokenizer.chat_template[:150]}...")
else:
    print("ATTENTION : pas de chat template !")

# Tokens speciaux
print(f"\nBOS token : {tokenizer.bos_token} (id={tokenizer.bos_token_id})")
print(f"EOS token : {tokenizer.eos_token} (id={tokenizer.eos_token_id})")
print(f"PAD token : {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"Vocab size : {tokenizer.vocab_size}")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jsboi\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Chat template trouve (longueur : 2507 chars)
Debut du template : {%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
...

BOS token : None (id=None)
EOS token : <|im_end|> (id=151645)
PAD token : <|endoftext|> (id=151643)
Vocab size : 151643


## 6. Configuration LoRA

Meme configuration que PT-02 (SFT) : LoRA rank 8, alpha 16, sur attention + MLP. Le DPO charge 2 modeaux (policy + reference), donc LoRA est **obligatoire** pour rester sous 8 Go VRAM.

**Note memoire** :
- Modele de base 4-bit : ~0.4 Go
- Policy LoRA (trainable) : ~15 Mo
- Reference (gelee, partage les poids de base) : ~0 Go supplementaire (shared embeddings)
- Total estime : ~1.5 Go avec 4-bit + LoRA

In [6]:
from peft import LoraConfig, TaskType

# Configuration LoRA (identique a PT-02 pour coherence)
LORA_CONFIG = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
        "gate_proj", "up_proj", "down_proj",      # MLP
    ],
)

print("Configuration LoRA :")
print(f"  rank = {LORA_CONFIG.r}")
print(f"  alpha = {LORA_CONFIG.lora_alpha}")
print(f"  dropout = {LORA_CONFIG.lora_dropout}")
print(f"  target_modules = {LORA_CONFIG.target_modules}")
print(f"  Type de tache = {LORA_CONFIG.task_type}")

Configuration LoRA :
  rank = 8
  alpha = 16
  dropout = 0.05
  target_modules = {'gate_proj', 'o_proj', 'up_proj', 'q_proj', 'k_proj', 'down_proj', 'v_proj'}
  Type de tache = TaskType.CAUSAL_LM


## 7. Configuration DPO

Hyperparametres cles du DPO :
- **beta = 0.1** : regularisation KL. Trop faible = le modele diverge de la reference. Trop eleve = pas d'apprentissage.
- **learning_rate = 5e-7** : plus bas que SFT (2e-4) car DPO ajuste subtilement les preferences, ne re-apprend pas tout.
- **per_device_train_batch_size = 1** : DPO charge 2 sequences par exemple (chosen + rejected) = double la memoire vs SFT.

In [7]:
DPO_CONFIG_DICT = {
    "beta": 0.1,                    # KL regularization strength
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,  # Effective batch = 16
    "learning_rate": 5e-7,          # Much lower than SFT
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.1,
    "max_length": 1024,            # TRL 1.7.0: max_seq_length renamed -> max_length (max_prompt_length removed)
    "logging_steps": 5,
    "save_strategy": "epoch",   # checkpoint chaque epoch (safe training anti-outage)
    "output_dir": "./dpo_output",
    "seed": 42,
    "bf16": True,
    "remove_unused_columns": False,
}

print("Configuration DPO :")
for k, v in DPO_CONFIG_DICT.items():
    print(f"  {k} = {v}")

Configuration DPO :
  beta = 0.1
  per_device_train_batch_size = 1
  gradient_accumulation_steps = 16
  learning_rate = 5e-07
  lr_scheduler_type = cosine
  warmup_ratio = 0.1
  max_seq_length = 1024
  max_prompt_length = 512
  logging_steps = 5
  save_strategy = no
  output_dir = ./dpo_output
  seed = 42
  bf16 = True
  remove_unused_columns = False


### Exercice 2 : Analyse de sensibilite au parametre beta

Le parametre `beta` controle la regularisation KL dans le loss DPO. L'objectif est d'implementer une fonction `simuler_dpo_loss` qui calcule la valeur du loss DPO pour differentes valeurs de beta, et de tracer la courbe de sensibilite.

**Objectif** : implementer le calcul du loss DPO (sigmoid du log-ratio) pour des valeurs de beta variant de 0.01 a 1.0, avec des log-ratios simules, puis afficher la courbe.

**Indices** :
- # Etape 1 : Simuler un ecart de log-probabilites entre chosen et rejected (par exemple 0.5)
- # Etape 2 : Calculer le loss = -log(sigmoid(beta * ecart)) pour chaque valeur de beta
- # Indice : utiliser `numpy` pour generer les valeurs de beta et `matplotlib` pour tracer

In [1]:
import numpy as np

def simuler_dpo_loss(log_ratio_diff: float = 0.5, beta_range: tuple = (0.01, 1.0), n_points: int = 50) -> list[tuple[float, float]]:
    # TODO etudiant : calculer le loss DPO pour differentes valeurs de beta
    
    betas = np.linspace(beta_range[0], beta_range[1], n_points)
    results = []
    
    for beta in betas:
        # Etape 1 : calculer l'argument de la sigmoid
        arg = None  # TODO etudiant : beta * log_ratio_diff
        
        # Etape 2 : calculer le loss = -log(sigmoid(arg))
        loss = None  # TODO etudiant : utiliser np.logaddexp pour stabilite numerique
        results.append((beta, loss))
    
    return results  # TODO etudiant : retourner la liste des (beta, loss)

print("Exercice a completer : sensibilite beta")

Exercice a completer


## 8. Construction du DPOTrainer

Le `DPOTrainer` de TRL encapsule toute la logique :
1. Charge le modele de base + applique LoRA
2. Cree une copie gelee comme reference (`ref_model`)
3. Tokenise les paires chosen/rejected via le chat template
4. Calcule le loss DPO sur chaque batch

**Pattern conditionnel** : `LOAD_MODEL_AND_TRAIN` controlle l'execution reelle.

In [8]:
if LOAD_MODEL_AND_TRAIN and CUDA_AVAILABLE:
    from transformers import AutoModelForImageTextToText, BitsAndBytesConfig
    from trl import DPOTrainer, DPOConfig
    from peft import get_peft_model

    # Quantization 4-bit
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    # Charger le modele de base
    print("Chargement du modele de base (4-bit)...")
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )

    # Appliquer LoRA
    print("Application LoRA...")
    model = get_peft_model(model, LORA_CONFIG)
    model.print_trainable_parameters()

    # Reference model : le DPOTrainer cree automatiquement une copie
    # du modele de base sans LoRA comme reference
    print("Configuration DPOConfig...")
    dpo_config = DPOConfig(**DPO_CONFIG_DICT)

    print("Construction DPOTrainer...")
    trainer = DPOTrainer(
        model=model,
        args=dpo_config,
        processing_class=tokenizer,
        train_dataset=dataset_dpo,
        # ref_model=None par defaut : le trainer cree la reference automatiquement
    )

    print(f"\nDPOTrainer pret. Dataset : {len(dataset_dpo)} exemples.")
    print("Lancement de l'entrainement...")
    train_result = trainer.train()
    print(f"\nEntrainement termine.")
    print(f"Pertes finales : {train_result.training_loss:.4f}")

else:
    print("Skip : entrainement DPO non execute (LOAD_MODEL_AND_TRAIN=False ou pas de CUDA)")
    print("Pour executer :")
    print("  1. Positionner LOAD_MODEL_AND_TRAIN = True dans la cellule 4")
    print("  2. Disposer d'un GPU avec >= 8 Go VRAM")
    print("  3. Re-executer cette cellule")
    print()
    print("Resultats attendus (RTX 3070, 50 exemples, ~10 min) :")
    print("  - Loss initial : ~0.69 (log(2), random)")
    print("  - Loss final : ~0.35-0.50 (convergence partielle)")
    print("  - Preference accuracy : ~65-75% sur train subset")

Skip : entrainement DPO non execute (LOAD_MODEL_AND_TRAIN=False ou pas de CUDA)
Pour executer :
  1. Positionner LOAD_MODEL_AND_TRAIN = True dans la cellule 4
  2. Disposer d'un GPU avec >= 8 Go VRAM
  3. Re-executer cette cellule

Resultats attendus (RTX 3070, 50 exemples, ~10 min) :
  - Loss initial : ~0.69 (log(2), random)
  - Loss final : ~0.35-0.50 (convergence partielle)
  - Preference accuracy : ~65-75% sur train subset


## 9. Evaluation : preference accuracy

La metrique cle du DPO est la **preference accuracy** : sur un ensemble de paires, a quelle frequence le modele assigne une probabilite plus elevee a chosen qu'a rejected ?

$$\text{PrefAcc} = \frac{1}{N} \sum_{i=1}^{N} \mathbb{1}\left[ \log \pi_\theta(y_w | x) > \log \pi_\theta(y_l | x) \right]$$

- PrefAcc = 0.50 = hasard (le modele ne distingue pas chosen de rejected)
- PrefAcc = 1.00 = parfait (mais potentiel surapprentissage)
- Zone cible sur petit dataset : 0.60-0.80

In [ ]:
# Evaluation de la preference accuracy sur le modele entraine
# (vraie evaluation : log-probabilites reelles calculees par le modele, PAS une simulation)
import torch.nn.functional as F


@torch.no_grad()
def _logprob_reponse(modele, tokenizer, prompt, reponse):
    """Log-probabilite moyenne par token de la reponse (normalisee par la longueur).

    On distingue le prefixe (prompt + marqueur de tour assistant) de la reponse
    pour ne sommer que les log-probs des tokens effectivement produits.
    """
    prefixe = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    complet = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt},
         {"role": "assistant", "content": reponse}], tokenize=False, add_generation_prompt=False)
    ids_prefixe = tokenizer(prefixe, add_special_tokens=False).input_ids
    ids_complet = tokenizer(complet, add_special_tokens=False).input_ids
    # tokens de la reponse = complet minus prefixe, sans le jeton final (eos/im_end)
    ids_reponse = ids_complet[len(ids_prefixe):-1]
    if len(ids_reponse) == 0:
        return 0.0
    entree = torch.tensor([ids_complet[:-1]]).to(modele.device)
    logits = modele(entree).logits[0].float()
    logp = F.log_softmax(logits, dim=-1)
    debut = len(ids_prefixe) - 1  # logits[debut] predit ids_complet[len(ids_prefixe)]
    cibles = torch.tensor(ids_reponse).to(modele.device)
    lp = logp[debut:debut + len(ids_reponse)].gather(1, cibles.unsqueeze(1))
    return lp.sum().item() / len(ids_reponse)  # normalise par la longueur


if LOAD_MODEL_AND_TRAIN and CUDA_AVAILABLE:
    print("Evaluation reelle du modele entraine sur un subset neutre (train_prefs[40:50])...")
    eval_dpo = load_dataset(
        "HuggingFaceH4/ultrafeedback_binarized", split="train_prefs[40:50]")
    correct = 0
    total = 0
    for ex in eval_dpo:
        prompt = ex["prompt"]
        choisi = ex["chosen"][-1]["content"]
        rejete = ex["rejected"][-1]["content"]
        lp_choisi = _logprob_reponse(model, tokenizer, prompt, choisi)
        lp_rejete = _logprob_reponse(model, tokenizer, prompt, rejete)
        correct += int(lp_choisi > lp_rejete)
        total += 1
    dpo_acc = correct / total if total else 0.0
    print()
    print("Preference accuracy reelle (modele entraine) :")
    print(f"  Baseline aleatoire                : 50.0%")
    print(f"  Apres DPO (modele entraine)       : {dpo_acc:.1%} ({correct}/{total} paires)")
    print(f"  Amelioration vs random            : +{(dpo_acc - 0.5) * 100:.1f} pp")
    print()
    print("Ces valeurs sont issues d'une VRAIE evaluation (forward pass du modele entraine),")
    print("pas d'une simulation. Verdict honnete (G.2) : sur 50 exemples d'entrainement,")
    print("l'amelioration est indicative — une evaluation multi-seed rigoureuse (>= 4 seeds)")
    print("est necessaire pour conclure (cf PT-06).")
else:
    print("Skip : evaluation non executee (LOAD_MODEL_AND_TRAIN=False ou pas de CUDA).")
    print("La preference accuracy reelle necessite l'entrainement complet (cellule 20).")


## 10. 4 pieges classiques du DPO en pratique

### Piege 1 : KL collapse (beta trop faible)
Si $\beta$ est trop petit, le modele peut diverger completement de la reference $\pi_{\text{ref}}$ — le log-ratio explose, le loss devient instable.
**Solution** : commencer avec $\beta = 0.1$, ajuster par facteur 2.

### Piege 2 : Reference drift (fine-tuner la reference)
Si la reference n'est pas **gelee** et partage des poids avec la policy (cas avec LoRA), les deux modeles convergent et le loss DPO tend vers zero sans apprentissage reel.
**Solution** : verifier que `ref_model.requires_grad_(False)` est bien applique. TRL le gere automatiquement.

### Piege 3 : Overfit sur beta (ajuster beta sur le validation set)
Optimiser $\beta$ sur le validation set = fuite d'information. Le beta controle le compromis alignement/performance, pas un hyperparametre a optimiser comme le learning rate.
**Solution** : fixer $\beta$ a priori (0.05-0.2) et le justifier theoriquement.

### Piege 4 : Format tokens et chat template incoherent
Si le SFT (PT-02) et le DPO n'utilisent pas le meme chat template, les log-probabilites sont incoherentes et le loss DPO est bruite.
**Solution** : verifier `tokenizer.chat_template` identique entre SFT et DPO. Utiliser le meme `MODEL_NAME`.

### Exercice 3 : Implementation de la preference accuracy

La section 9 presente la formule de la preference accuracy mais l'evaluation est simulee. L'objectif est d'implementer la fonction `preference_accuracy` qui compare les log-probabilites de deux modeles (policy vs reference) sur des paires chosen/rejected.

**Objectif** : ecrire une fonction qui, pour chaque paire, determine si le modele assigne une probabilite plus elevee a chosen qu'a rejected, et retourne le taux de succes global.

**Indices** :
- # Etape 1 : Pour chaque paire, recuperer les log-probabilites de chosen et rejected
- # Etape 2 : Comparer les log-probabilites normalisees (par la longueur de la sequence)
- # Indice : en mode CPU-safe, simuler les log-probs avec des valeurs aleatoires pour tester la logique

In [1]:
def preference_accuracy(
    log_probs_chosen: list[float],
    log_probs_rejected: list[float],
) -> float:
    # TODO etudiant : calculer la preference accuracy
    # log_probs_chosen[i] = log P(chosen_i | prompt_i)
    # log_probs_rejected[i] = log P(rejected_i | prompt_i)
    
    # Etape 1 : comparer chaque paire
    correct = 0
    for lp_chosen, lp_rejected in zip(log_probs_chosen, log_probs_rejected):
        pass  # TODO etudiant : incrementer si chosen > rejected
    
    # Etape 2 : calculer le taux
    accuracy = None  # TODO etudiant : correct / total
    return accuracy

print("Exercice a completer : preference accuracy")

Exercice a completer


## 11. Methodologie multi-seed (bonus)

Pour toute claim "DPO > SFT" sur une metrique d'evaluation, la regle CoursIA exige :
- **>= 4 seeds** parmi {0, 1, 7, 42, 99}
- **Edge >= 2 sigma** cross-seed (sinon flag "noise")
- **Walk-forward 5-fold** sur les donnees de preference

En pratique pour ce notebook pedagogique (50 exemples), un verdict honnete serait :
- "DPO montre une amelioration sur 3/4 seeds, mais edge < 2 sigma"
- "INCONCLUSIF sur ce subset — a confirmer sur dataset complet"

Le pattern honnete (G.2) : rapporter le verdict reel, pas le verdict souhaite.

## 12. Transition vers PT-04 — GRPO (Deepseek-R1)

Le DPO optimise directement sur des **paires de preference** — mais il necessite encore des donnees humaines (ou synthetiques) de preference. Le GRPO (PT-04) va plus loin :

| Aspect | DPO | GRPO |
|--------|-----|------|
| Donnees requises | Paires chosen/rejected | Prompts + reward function |
| Reward model | Non (implicite) | Non (function exacte) |
| Memoire | 2 modeles (policy + ref) | 1 modele + G outputs par prompt |
| Origine | Rafailov 2023 (Stanford) | Shao 2024 (Deepseek) |
| Cas d'usage | Alignement instruction-following | Reasoning emergent (math, code) |

Le GRPO = la technique cle derriere **Deepseek-R1** (janvier 2025) — le prochain notebook (PT-04) sera le livrable central de cette serie.

**Pourquoi GRPO > PPO pour le reasoning** : pas de critic network, advantage calcule par normalisation intra-group (variance naturelle reduite), recompense verifiable (exact match sur math/code).